In [5]:
# Importing libraries

import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [6]:
# Getting the data

import pandas as pd
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
df = pd.read_csv(url)

In [7]:
# Previewing the first 10 rows of the dataset to make sure it's working properly
df.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [8]:
"""
Important Information and Dataset Columns:

date: the date of the match
home_team and away_team: names of the competing teams
home_score and away_score: full_time home/away team score including extra time, not including penalty shootouts
tournament: name of the tournament
city: name of the administrative unit where the match was played
country: name of the country where the match was played
neutral: TRUE/FALSE column indicating whether the match was in a neutral venue
"""

'\nImportant Information and Dataset Columns:\n\ndate: the date of the match\nhome_team and away_team: names of the competing teams\nhome_score and away_score: full_time home/away team score including extra time, not including penalty shootouts\ntournament: name of the tournament\ncity: name of the administrative unit where the match was played\ncountry: name of the country where the match was played\nneutral: TRUE/FALSE column indicating whether the match was in a neutral venue\n'

In [ ]:
# Calculating the Home Field Advantage (HFA)

elo_dist = {}

# Sort the dataframe chronologically
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Define Mathematical Helpers
def calc_g_multiplier(goal_diff):
    gd = abs(goal_diff)
    if gd <= 1:
        return 1.0
    elif gd == 2:
        return 1.5
    else:
        return (11.0 + gd) / 8.0

def get_k_factor(tournament_name):
    t = str(tournament_name).lower()
    if 'friendly' in t:
        return 20
    elif 'qualification' in t:
        return 40
    elif 'world cup' in t and 'qualification' not in t:
        return 60
    elif any(x in t for x in ['euro', 'copa américa', 'african cup', 'asian cup', 'gold cup']):
        return 50
    else:
        return 30

# 3. Initialize the global dictionary
elo_dist = {}

# 4. The Time-Series Loop
for row in df.itertuples():
    home = row.home_team
    away = row.away_team
    h_score = row.home_score
    a_score = row.away_score
    
    # Skip rows with missing data
    if pd.isna(h_score) or pd.isna(a_score):
        continue
        
    # Initialize unseen teams
    if home not in elo_dist: elo_dist[home] = 1500.0
    if away not in elo_dist: elo_dist[away] = 1500.0
    
    r_home = elo_dist[home]
    r_away = elo_dist[away]
    
    # Apply Home Field Advantage (+100) if not a neutral venue
    r_home_adjusted = r_home + 100.0 if not row.neutral else r_home
    
    # Calculate Expected Win Probability
    e_home = 1.0 / (1.0 + 10.0 ** ((r_away - r_home_adjusted) / 400.0))
    e_away = 1.0 - e_home
    
    # Determine Actual Result (S)
    if h_score > a_score:
        s_home, s_away = 1.0, 0.0
    elif h_score < a_score:
        s_home, s_away = 0.0, 1.0
    else:
        s_home, s_away = 0.5, 0.5
        
    # Get Weights
    k = get_k_factor(row.tournament)
    g = calc_g_multiplier(h_score - a_score)
    
    # Calculate Points Exchanged
    point_change_home = k * g * (s_home - e_home)
    point_change_away = k * g * (s_away - e_away)
    
    # Update global dictionary
    elo_dist[home] += point_change_home
    elo_dist[away] += point_change_away

# 5. Filter for the 48 Qualified Teams for the 2026 World Cup
wc_2026_teams = [
    # Group A
    "Mexico", "South Africa", "South Korea", "Czech Republic", 
    # Group B
    "Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina", 
    # Group C
    "Brazil", "Morocco", "Haiti", "Scotland", 
    # Group D
    "United States", "Paraguay", "Australia", "Turkey", 
    # Group E
    "Germany", "Curaçao", "Ivory Coast", "Ecuador", 
    # Group F
    "Netherlands", "Japan", "Tunisia", "Sweden", 
    # Group G
    "Belgium", "Egypt", "Iran", "New Zealand", 
    # Group H
    "Spain", "Cape Verde", "Saudi Arabia", "Uruguay", 
    # Group I
    "France", "Senegal", "Norway", "Iraq", 
    # Group J
    "Argentina", "Algeria", "Austria", "Jordan", 
    # Group K
    "Portugal", "Uzbekistan", "Colombia", "DR Congo", 
    # Group L
    "England", "Croatia", "Ghana", "Panama"
]

# Create the final DataFrame
wc_elo_data = {team: elo_dist.get(team, None) for team in wc_2026_teams}
df_wc_elo = pd.DataFrame(list(wc_elo_data.items()), columns=['Team', 'Pre_Tournament_Elo'])

# Sort from strongest to weakest to view Power Rankings
df_wc_elo = df_wc_elo.sort_values('Pre_Tournament_Elo', ascending=False).reset_index(drop=True)
df_wc_elo.head(10)

,Team,Pre_Tournament_Elo
0,Spain,2215.083283
1,Argentina,2183.778460
2,France,2136.465730
3,England,2077.687334
4,Brazil,2056.406045
5,Colombia,2053.307748
6,Portugal,2021.819790
7,Netherlands,2020.874972
8,Ecuador,2018.371496
9,Germany,1993.819477


In [13]:
# Importing the optimizer

# 1. Wrap your Elo logic inside a function that returns the Brier Score
def evaluate_elo_model(kf, kq, kc, kwc):
    # Create a fresh dictionary for every simulation run
    temp_elo_dist = {}
    brier_scores = []
    
    for row in df.itertuples():
        home = row.home_team
        away = row.away_team
        h_score = row.home_score
        a_score = row.away_score
        
        if pd.isna(h_score) or pd.isna(a_score):
            continue
            
        if home not in temp_elo_dist: temp_elo_dist[home] = 1500.0
        if away not in temp_elo_dist: temp_elo_dist[away] = 1500.0
        
        r_home = temp_elo_dist[home]
        r_away = temp_elo_dist[away]
        
        r_home_adjusted = r_home + 100.0 if not row.neutral else r_home
        
        e_home = 1.0 / (1.0 + 10.0 ** ((r_away - r_home_adjusted) / 400.0))
        e_away = 1.0 - e_home
        
        if h_score > a_score:
            s_home, s_away = 1.0, 0.0
        elif h_score < a_score:
            s_home, s_away = 0.0, 1.0
        else:
            s_home, s_away = 0.5, 0.5
            
        # Dynamically assign the K-factor based on our optimizer's current guess
        t = str(row.tournament).lower()
        if 'friendly' in t:
            k = kf
        elif 'qualification' in t:
            k = kq
        elif 'world cup' in t and 'qualification' not in t:
            k = kwc
        elif any(x in t for x in ['euro', 'copa américa', 'african cup', 'asian cup', 'gold cup']):
            k = kc
        else:
            k = 30.0 # Default for minor tournaments
            
        g = calc_g_multiplier(h_score - a_score)
        
        # Track accuracy (Brier Score) ONLY for modern matches (e.g., Year 2000 onwards)
        if row.date.year >= 2000:
            brier_scores.append((e_home - s_home) ** 2)
            
        # Update ratings
        point_change_home = k * g * (s_home - e_home)
        point_change_away = k * g * (s_away - e_away)
        
        temp_elo_dist[home] += point_change_home
        temp_elo_dist[away] += point_change_away
        
    return np.mean(brier_scores)

# 2. Define the objective function for SciPy
def objective(params):
    kf, kq, kc, kwc = params
    
    # Add penalties to stop the optimizer from testing illogical weights
    if kf < 0 or kq < 0 or kc < 0 or kwc < 0:
        return 999.0
    if kf >= kwc: # A friendly cannot be worth more than a World Cup match
        return 999.0
        
    score = evaluate_elo_model(kf, kq, kc, kwc)
    print(f"Testing K's: Fr={kf:.1f}, Qual={kq:.1f}, Cont={kc:.1f}, WC={kwc:.1f} | Brier: {score:.5f}")
    return score

# 3. Run the optimization
initial_guess = [20.0, 40.0, 50.0, 60.0]

print("Starting optimization... (This will simulate the 150-year history dozens of times. It may take a few minutes.)")
result = minimize(
    objective, 
    initial_guess, 
    method='Nelder-Mead',
    options={'maxiter': 50} # Capped at 50 iterations so your notebook doesn't hang forever
)

print("\n=== OPTIMAL K-FACTORS FOUND ===")
print(f"Friendly: {result.x[0]:.2f}")
print(f"Qualifier: {result.x[1]:.2f}")
print(f"Continental: {result.x[2]:.2f}")
print(f"World Cup: {result.x[3]:.2f}")

Starting optimization... (This will simulate the 150-year history dozens of times. It may take a few minutes.)
Testing K's: Fr=20.0, Qual=40.0, Cont=50.0, WC=60.0 | Brier: 0.13337
Testing K's: Fr=21.0, Qual=40.0, Cont=50.0, WC=60.0 | Brier: 0.13330
Testing K's: Fr=20.0, Qual=42.0, Cont=50.0, WC=60.0 | Brier: 0.13347
Testing K's: Fr=20.0, Qual=40.0, Cont=52.5, WC=60.0 | Brier: 0.13342
Testing K's: Fr=20.0, Qual=40.0, Cont=50.0, WC=63.0 | Brier: 0.13338
Testing K's: Fr=20.5, Qual=38.0, Cont=51.2, WC=61.5 | Brier: 0.13328
Testing K's: Fr=20.8, Qual=36.0, Cont=51.9, WC=62.2 | Brier: 0.13321
Testing K's: Fr=20.9, Qual=38.0, Cont=48.4, WC=62.6 | Brier: 0.13320
Testing K's: Fr=21.3, Qual=37.0, Cont=46.4, WC=63.9 | Brier: 0.13310
Testing K's: Fr=21.5, Qual=36.5, Cont=49.1, WC=60.1 | Brier: 0.13310
Testing K's: Fr=22.3, Qual=34.8, Cont=48.7, WC=63.1 | Brier: 0.13301
Testing K's: Fr=23.4, Qual=32.1, Cont=48.1, WC=64.7 | Brier: 0.13291
Testing K's: Fr=22.5, Qual=30.8, Cont=47.7, WC=65.5 | Brier: 

In [14]:
# Rerunning the algorithm with the optimized K-factors

elo_dist = {}

# Sort the dataframe chronologically
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Define Mathematical Helpers
def calc_g_multiplier(goal_diff):
    gd = abs(goal_diff)
    if gd <= 1:
        return 1.0
    elif gd == 2:
        return 1.5
    else:
        return (11.0 + gd) / 8.0

def get_k_factor(tournament_name):
    t = str(tournament_name).lower()
    if 'friendly' in t:
        return 27.09
    elif 'qualification' in t:
        return 30.77
    elif 'world cup' in t and 'qualification' not in t:
        return 37.61
    elif any(x in t for x in ['euro', 'copa américa', 'african cup', 'asian cup', 'gold cup']):
        return 27.41
    else:
        return 30

# 3. Initialize the global dictionary
elo_dist = {}

# 4. The Time-Series Loop
for row in df.itertuples():
    home = row.home_team
    away = row.away_team
    h_score = row.home_score
    a_score = row.away_score
    
    # Skip rows with missing data
    if pd.isna(h_score) or pd.isna(a_score):
        continue
        
    # Initialize unseen teams
    if home not in elo_dist: elo_dist[home] = 1500.0
    if away not in elo_dist: elo_dist[away] = 1500.0
    
    r_home = elo_dist[home]
    r_away = elo_dist[away]
    
    # Apply Home Field Advantage (+100) if not a neutral venue
    r_home_adjusted = r_home + 100.0 if not row.neutral else r_home
    
    # Calculate Expected Win Probability
    e_home = 1.0 / (1.0 + 10.0 ** ((r_away - r_home_adjusted) / 400.0))
    e_away = 1.0 - e_home
    
    # Determine Actual Result (S)
    if h_score > a_score:
        s_home, s_away = 1.0, 0.0
    elif h_score < a_score:
        s_home, s_away = 0.0, 1.0
    else:
        s_home, s_away = 0.5, 0.5
        
    # Get Weights
    k = get_k_factor(row.tournament)
    g = calc_g_multiplier(h_score - a_score)
    
    # Calculate Points Exchanged
    point_change_home = k * g * (s_home - e_home)
    point_change_away = k * g * (s_away - e_away)
    
    # Update global dictionary
    elo_dist[home] += point_change_home
    elo_dist[away] += point_change_away

# 5. Filter for the 48 Qualified Teams for the 2026 World Cup
wc_2026_teams = [
    # Group A
    "Mexico", "South Africa", "South Korea", "Czech Republic", 
    # Group B
    "Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina", 
    # Group C
    "Brazil", "Morocco", "Haiti", "Scotland", 
    # Group D
    "United States", "Paraguay", "Australia", "Turkey", 
    # Group E
    "Germany", "Curaçao", "Ivory Coast", "Ecuador", 
    # Group F
    "Netherlands", "Japan", "Tunisia", "Sweden", 
    # Group G
    "Belgium", "Egypt", "Iran", "New Zealand", 
    # Group H
    "Spain", "Cape Verde", "Saudi Arabia", "Uruguay", 
    # Group I
    "France", "Senegal", "Norway", "Iraq", 
    # Group J
    "Argentina", "Algeria", "Austria", "Jordan", 
    # Group K
    "Portugal", "Uzbekistan", "Colombia", "DR Congo", 
    # Group L
    "England", "Croatia", "Ghana", "Panama"
]

# Note: Depending on the dataset's exact naming conventions, you might need to adjust 
# names like "South Korea" to "Korea Republic" or "DR Congo" to "Congo DR".

# 6. Create the final DataFrame
wc_elo_data = {team: elo_dist.get(team, None) for team in wc_2026_teams}
df_wc_elo = pd.DataFrame(list(wc_elo_data.items()), columns=['Team', 'Pre_Tournament_Elo'])

# Sort from strongest to weakest to view your Power Rankings
df_wc_elo = df_wc_elo.sort_values('Pre_Tournament_Elo', ascending=False).reset_index(drop=True)
df_wc_elo.head(10)

,Team,Pre_Tournament_Elo
0,Spain,2166.134956
1,Argentina,2163.137504
2,France,2119.621410
3,Brazil,2063.820766
4,Colombia,2039.144479
5,Portugal,2031.077085
6,England,2030.112162
7,Netherlands,2008.254064
8,Ecuador,1987.863760
9,Japan,1983.047226


In [15]:
# Saving the new elo ratings into a DataFrame
df_wc_elo.to_csv("optimized_elo_2026.csv", index=False)